# CNN for Text Classification
This notebook uses the TextCNN from Kim (2014) to classify FinancialPhraseBank (Malo et al., 2014).

Topics covered:

(1) Building and loading a custom dataset

(2) Loading and using pretrained word2vec

(3) Building the TextCNN



## 1. Data Loading and Preprocessing

In [1]:
with open('data/FinancialPhraseBank/Sentences_75Agree.txt', 'r') as f: 
    FPB_75_in = f.read() # FinancialPhraseBank ships subsets at different annotator-agreement levels
with open('data/FinancialPhraseBank/Sentences_AllAgree.txt', 'r') as f:
    FPB_100_in = f.read()

FileNotFoundError: [Errno 2] No such file or directory: 'data/FinancialPhraseBank/Sentences_75Agree.txt'

In [ ]:
FPB_75_in[:200]

'According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .@neutral\nWith the new production plant the company would increase its cap'

In [ ]:
FPB_75_lines = FPB_75_in.split('\n')

In [ ]:
FPB_75_lines[0].split('@')

['According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .',
 'neutral']

In [ ]:
from string import punctuation

FPB_75_data = []
FPB_75_label = []
tags = {'negative':0,'neutral':1,'positive':2}
for line in FPB_75_lines[:-1]:
    line = line.lower()
    data_line = line.split('@')[0]
    FPB_75_label.append(tags[line.split('@')[1]])
    data_line = ''.join([c for c in data_line if c not in punctuation]) # remove punctuation
    data_l = data_line.split(' ')
    data_l = [c for c in data_l if len(c)>0] # punctuation here is space-separated, so splitting on spaces after removing it leaves some empty tokens
    FPB_75_data.append(data_l)
    

In [ ]:
print(FPB_75_data[0],'\n',FPB_75_label[0])

['according', 'to', 'gran', 'the', 'company', 'has', 'no', 'plans', 'to', 'move', 'all', 'production', 'to', 'russia', 'although', 'that', 'is', 'where', 'the', 'company', 'is', 'growing'] 
 1


In [ ]:
FPB_100_lines = FPB_100_in.split('\n')
FPB_100_data = []
FPB_100_label = []
tags = {'negative':0,'neutral':1,'positive':2}
for line in FPB_100_lines[:-1]:
    line = line.lower()
    data_line = line.split('@')[0]
    FPB_100_label.append(tags[line.split('@')[1]])
    data_line = ''.join([c for c in data_line if c not in punctuation])
    data_l = data_line.split(' ')
    data_l = [c for c in data_l if len(c)>0]
    FPB_100_data.append(data_l)

In [ ]:
FPB_data_raw = FPB_75_data + FPB_100_data
FPB_label_raw = FPB_75_label + FPB_100_label

In [ ]:
FP_len = []
for li in FPB_data_raw:
    FP_len.append(len(li))

import pandas as pd

pd.DataFrame(FP_len).describe()

,0
count,5717.000000
mean,20.027287
std,8.876611
min,1.000000
25%,13.000000
50%,19.000000
75%,26.000000
max,52.000000


In [ ]:
import random 
random.seed(42)
raw_idx = list(range(len(FPB_data_raw)))

In [ ]:
random.shuffle(raw_idx)


In [ ]:
FPB_data = []
FPB_label = []
for idx in raw_idx:
    FPB_data.append(FPB_data_raw[idx])
    FPB_label.append(FPB_label_raw[idx])

In [ ]:
FPB_label_raw[0]

1

## 2. Loading the Pretrained word2vec Model

In [ ]:
# load the word2vec model via gensim
from gensim.models import KeyedVectors
# use the SLIM model; the full version uses too much memory/VRAM
# download: https://github.com/eyaler/word2vec-slim/blob/master/GoogleNews-vectors-negative300-SLIM.bin.gz
embed_lookup = KeyedVectors.load_word2vec_format('model/GoogleNews-vectors-negative300-SLIM.bin', 
                                                 binary=True)

In [ ]:
embed_lookup.key_to_index['hello']

18680

In [ ]:
def get_vec_index(wordl): # return the word2vec index for each token
    resl = []
    for wi in wordl:
        try:
            idx = embed_lookup.key_to_index[wi]
        except Exception:
            idx = 0
        resl.append(idx)
    return resl

In [ ]:
FPB_vec_index = list(map(get_vec_index,FPB_data))
# FPB_vec_index = [get_vec_index(wi) for wi in FPB_data]  # a list comprehension is the more idiomatic form

In [ ]:
import numpy as np
def padding_tokens(raw_vec,padding_len=60, sq_len=None): # pad all samples; here every sample is shorter than 60, but other cases may need truncation
    tlen = len(raw_vec)
    if sq_len is not None:
        sq_len.append(tlen)
    if tlen>=padding_len:
        return raw_vec[:padding_len]
    
    for li in range(padding_len-tlen):
        raw_vec.append(0)
    return np.array(raw_vec)

In [ ]:
sq_len = []
FPB_vec_index_padding = list(map(lambda x: padding_tokens(x, padding_len=60, sq_len=sq_len),FPB_vec_index))
# this could also be written as a list comprehension for better readability

In [ ]:
features = np.array(FPB_vec_index_padding)

In [ ]:
labels = np.array(FPB_label)

In [ ]:
features[:10,:6]

array([[   796,    864,    563,      0,  17001,      0],
       [   468,      4,    584, 178363,      8,      0],
       [     0,   6638,      0,     10,      9,     54],
       [     9,   1179,    130,    840,      0,    682],
       [182879,    730,      0,      0,   1239,   6407],
       [     9,     94,   1006,      3,      0,      0],
       [     9,    429,      3,    249,      0,     14],
       [     0,      1,      9,     32,      0,   1470],
       [     0,      0,      0,      0,    864,     93],
       [   172,    796,    864,      0,   5119,      1]])

## 3. Standardizing the Data

In [ ]:
# split into train / validation / test sets
train_frac = 0.8
valid_frac = 0.9

train_idx = int(len(features)*train_frac)
valid_idx = int(len(features)*valid_frac)

train_x, valid_x,test_x = features[:train_idx], features[train_idx:valid_idx], features[valid_idx:]
train_y, valid_y,test_y = labels[:train_idx], labels[train_idx:valid_idx], labels[valid_idx:]
train_sq_len, valid_sq_len, test_sq_len = sq_len[:train_idx], sq_len[train_idx:valid_idx], sq_len[valid_idx:]


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# build standard TensorDatasets of features and labels
train_data = TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y).long())
valid_data = TensorDataset(torch.from_numpy(valid_x), torch.from_numpy(valid_y).long())
test_data = TensorDataset(torch.from_numpy(test_x), torch.from_numpy(test_y).long())

batch_size = 50

# shuffle and batch
train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)
valid_loader = DataLoader(valid_data, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_data, shuffle=True, batch_size=batch_size)

In [ ]:
# save the data for later use
torch.save({"data": train_data.tensors[0], "labels": train_data.tensors[1], "lengths": torch.tensor(train_sq_len)}, "data/Malo/Malo_Train.pth")
torch.save({"data": valid_data.tensors[0], "labels": valid_data.tensors[1], "lengths": torch.tensor(valid_sq_len)}, "data/Malo/Malo_Valid.pth")
torch.save({"data": test_data.tensors[0], "labels": test_data.tensors[1], "lengths": torch.tensor(test_sq_len)}, "data/Malo/Malo_Test.pth")

In [ ]:
len(valid_x)

572

In [ ]:
# inspect the DataLoader output
for batch_idx, (x, y) in enumerate(train_loader):
    print(f"Train set - Batch {batch_idx}")
    print(f"Features shape: {x.shape}")  # [batch_size, word_len]
    print(f"Labels shape: {y.shape}")    # [batch_size]
    break  # one batch is enough to check

## 4. Defining the Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TextCNN(nn.Module):
    
    def __init__(self, embed_model, vocab_size, output_size, embed_dim,    # arguments required when instantiating TextCNN
                 num_filters=100, kernel_sizes=[3, 4, 5], embed_freeze=True, drop_prob=0.5):
        
        super(TextCNN, self).__init__()

        # hyperparameters
        self.num_filters = num_filters    # number of CNN channels (filters)
        self.embed_dim = embed_dim        # dimension of the word vectors
        
        # 1. embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # load the pretrained weights
        self.embedding.weight = nn.Parameter(torch.from_numpy(embed_model.vectors)) # all vectors
        # freeze the embedding layer; set to False to train it jointly
        if embed_freeze:
            self.embedding.requires_grad = False
        
        # 2. convolutional layers
        self.convs_1d = nn.ModuleList([
            nn.Conv2d(1, num_filters, (k, embed_dim), padding=(k-2,0))  # parallel 2D convolutions; each kernel spans the full embedding width (embed_dim)
            for k in kernel_sizes])
        
        # 3. fully connected layer
        self.fc = nn.Linear(len(kernel_sizes) * num_filters, output_size) 
        
        # 4. dropout
        self.dropout = nn.Dropout(drop_prob)
        
        
    
    def conv_and_pool(self, x, conv):  # helper for pooling and reshaping after a convolution
       
        x = F.relu(conv(x)).squeeze(3)
        # after each conv the 4th dim is 1 and can be squeezed out (dims: (batch_size, num_filters, sequence_length, 1))
        x_max = F.max_pool1d(x, x.size(2)).squeeze(2)
        # 1D max-pool over the 3rd dim (sequence_length); result is (batch_size, num_filters, 1), then squeeze it
        return x_max
        # returned shape is (batch_size, num_filters)

    def forward(self, x):
        
        # map token indices to embedding vectors
        embeds = self.embedding(x) 
        # input x: (batch_size, seq_length), where seq_length is the input sequence length
        # output embeds: (batch_size, seq_length, embedding_dim)
        # a 3D tensor: each token index is looked up in the word2vec table
        
        # unsqueeze(1) inserts a size-1 channel dim to match the conv input requirement
        embeds = embeds.unsqueeze(1)
        # input shape embeds: (batch_size, seq_length, embedding_dim)
        # output shape embeds: (batch_size, 1, seq_length, embedding_dim); 1 = single channel
        
        # list holding each conv's output
        conv_results = [self.conv_and_pool(embeds, conv) for conv in self.convs_1d]
        # self.convs_1d is a ModuleList of 2D convolutions, each with a different window size (kernel_size)
        # for each kernel conv:
        #    apply the convolution and extract features via self.conv_and_pool
        #    each kernel produces a feature map of shape (batch_size, num_filters)
    	# conv_results ends up as a list of every kernel's output

        # concatenate the conv outputs
        x = torch.cat(conv_results, 1)
        # apply dropout
        x = self.dropout(x)

        # fully connected layer
        x = self.fc(x)

        # softmax output
        return F.log_softmax(x, dim=1)
      

## 5. Training and Test Functions

In [ ]:
# the same train/test functions can be reused (still a classification task)
# define the training function
def train(args, model, device, train_loader, optimizer, epoch):
    # args: object/namespace of hyperparameters (e.g. batch_size, learning_rate) controlling training
    # model: the neural network to be trained
    # device: where model and data run (cpu or cuda)
    # train_loader: yields batches of training data
    # optimizer: updates model parameters (e.g. SGD, Adam)
    # epoch: current epoch number, for progress logging
    
    model.train()  # switch to training mode; dropout/BatchNorm behave differently in train vs eval
    for batch_idx, (data, target) in enumerate(train_loader):  # load a batch: data and target are inputs and labels; batch_idx starts at 0
        data, target = data.to(device), target.to(device)  # move data and labels to the device (CPU or GPU)
        optimizer.zero_grad()     # clear accumulated gradients; PyTorch accumulates by default, so reset before each step
        output = model(data)      # forward pass on the current batch
        loss = F.nll_loss(output, target)  # compute the loss; F.nll_loss is the negative log-likelihood loss
        loss.backward()           # backpropagate the loss
        optimizer.step()          # update parameters with the computed gradients

# define the test function
def test(model, device, test_loader):
    # model: the network to evaluate
    # device: where model and data run (CPU or GPU)
    # test_loader: provides the test set (usually unshuffled)
    
    model.eval()  # switch to eval mode
    test_loss = 0
    correct = 0
    with torch.no_grad():  # disable autograd to save memory and speed things up
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)    # move data and labels to the device (CPU or GPU)
            output = model(data)    # forward pass to get predictions
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # accumulate the NLL loss over the batch
            pred = output.argmax(dim=1, keepdim=True)   # take the argmax over classes as the prediction
                                                        # argmax(dim=1): index of the highest-probability class per row
                                                        # keepdim=True: keep dims aligned for comparison with labels
            correct += pred.eq(target.view_as(pred)).sum().item()  # count correct predictions; pred.eq(...) compares predictions with labels

    test_loss /= len(test_loader.dataset)  # average loss = total loss / number of test samples

    print("=" * 50)
    print("Evaluation Results:")
    print(f" - Average Loss: {test_loss:.4f}")
    print(f" - Accuracy: {correct}/{len(test_loader.dataset)} "
          f"({100. * correct / len(test_loader.dataset):.2f}%)")
    print("=" * 50)


In [ ]:
epochs = 6  # max number of training epochs
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # choose device: GPU (CUDA) if available, else CPU
# device = "cpu"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

mps


In [ ]:
pretrained_words = []
for word in embed_lookup.key_to_index:
    pretrained_words.append(word)

In [ ]:

vocab_size = len(pretrained_words)
output_size = 3 
embedding_dim = len(embed_lookup[pretrained_words[0]]) # 300-dim vectors
num_filters = 100
kernel_sizes = [3, 4, 5]

model = TextCNN(embed_lookup, vocab_size, output_size, embedding_dim,
                   num_filters, kernel_sizes).to(device)
print(model)


TextCNN(
  (embedding): Embedding(299567, 300)
  (convs_1d): ModuleList(
    (0): Conv2d(1, 100, kernel_size=(3, 300), stride=(1, 1), padding=(1, 0))
    (1): Conv2d(1, 100, kernel_size=(4, 300), stride=(1, 1), padding=(2, 0))
    (2): Conv2d(1, 100, kernel_size=(5, 300), stride=(1, 1), padding=(3, 0))
  )
  (fc): Linear(in_features=300, out_features=3, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [ ]:
import torch.optim as optim
optimizer = optim.Adadelta(model.parameters()) # Adadelta adapts the learning rate automatically; no global LR needed

## 6. Run

In [ ]:
import time
for epoch in range(1, epochs + 1):
    start_time = time.time()  # record start time
    train([], model, device, train_loader, optimizer, epoch)  # train
    train_time = time.time()
    test(model, device, valid_loader)  # test
    test_time = time.time()
    train_duration = train_time - start_time
    test_duration = test_time - train_time
    print(f"Epoch {epoch}/{epochs} - Trained in {train_duration:.2f} seconds  - Test in {test_duration:.2f} seconds.\n")


Evaluation Results:
 - Average Loss: 0.6390
 - Accuracy: 451/572 (78.85%)
Epoch 1/6 - Trained in 75.07 seconds  - Test in 0.55 seconds.



In [ ]:
test(model, device, valid_loader)

Evaluation Results:
 - Average Loss: 0.1838
 - Accuracy: 538/572 (94.06%)


In [ ]:
torch.save(model.state_dict(), "model/TextCNN.pth")